<a href="https://colab.research.google.com/github/nbrookes80-spec/LS_Based_Engine_Repository/blob/master/Face_Swap_Studio_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Face-Swap Studio - Colab Backend

GPU-accelerated face swapping engine using roop + insightface

In [19]:
# SECTION 1: Environment Setup
!pip install -q cython numpy
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu
!pip install -q insightface==0.7.3
!pip install -q pyzmq pillow opencv-python-headless gdown

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/ColabInputs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/ColabOutputs', exist_ok=True)

print("✓ Packages installed")
print("✓ Google Drive mounted")
print("✓ Output directories created")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Packages installed
✓ Google Drive mounted
✓ Output directories created


In [20]:
# SECTION 2: ZMQ Server Setup
import zmq
from threading import Thread
from collections import deque
import time

# Create context and sockets
context = zmq.Context()
job_socket = context.socket(zmq.PULL)
job_socket.bind("tcp://0.0.0.0:5556")
print("✓ Job socket bound to tcp://0.0.0.0:5556")

progress_socket = context.socket(zmq.PUSH)
progress_socket.bind("tcp://0.0.0.0:5557")
print("✓ Progress socket bound to tcp://0.0.0.0:5557")

health_socket = context.socket(zmq.REP)
health_socket.bind("tcp://0.0.0.0:5558")
print("✓ Health socket bound to tcp://0.0.0.0:5558")

# Create job queue
job_queue = deque()
is_running = True

# Job listener function
def listen_jobs():
    while is_running:
        try:
            job_data = job_socket.recv_json(flags=zmq.NOBLOCK)
            job_queue.append(job_data)
        except zmq.error.Again:
            pass
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(0.01)

# Health check function
def health_check():
    while is_running:
        try:
            message = health_socket.recv(flags=zmq.NOBLOCK)
            health_socket.send_json({"status": "ok"})
        except zmq.error.Again:
            pass
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(0.1)

# Start threads
job_thread = Thread(target=listen_jobs, daemon=True)
job_thread.start()
health_thread = Thread(target=health_check, daemon=True)
health_thread.start()

print("✓✓✓ COLAB ZMQ SERVER READY")

✓ Job socket bound to tcp://0.0.0.0:5556
✓ Progress socket bound to tcp://0.0.0.0:5557
✓ Health socket bound to tcp://0.0.0.0:5558
✓✓✓ COLAB ZMQ SERVER READY


In [21]:
# Install GFPGAN and its runtime dependencies without compiling the problematic legacy basicsr package
!pip install -q facexlib lmdb
!pip install -q gfpgan --no-deps

In [22]:
# SECTION 3: Face Swap Engine (insightface inswapper_128)
import cv2
import numpy as np
import onnxruntime as ort
from insightface.app import FaceAnalysis
from insightface.model_zoo import get_model

available = ort.get_available_providers()
print("ONNX providers:", available)

PROVIDERS = [p for p in ['CUDAExecutionProvider', 'CPUExecutionProvider'] if p in available]
CTX_ID = 0 if 'CUDAExecutionProvider' in available else -1

# Face detection + embeddings (downloads buffalo_l on first run)
face_analyzer = FaceAnalysis(name='buffalo_l', providers=PROVIDERS)
face_analyzer.prepare(ctx_id=CTX_ID, det_size=(640, 640))

# Face swapper (downloads inswapper_128.onnx, about 530 MB, on first run)
swapper = get_model('inswapper_128.onnx', download=True, download_zip=False, providers=PROVIDERS)

# Optional: GFPGAN restoration for quality='high'
restorer = None
try:
    from gfpgan import GFPGANer
    gfpgan_path = '/content/GFPGANv1.4.pth'
    if not os.path.exists(gfpgan_path):
        !wget -q -O /content/GFPGANv1.4.pth https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth
    restorer = GFPGANer(model_path=gfpgan_path, upscale=1, arch='clean',
                        channel_multiplier=2, bg_upsampler=None)
    print("✓ GFPGAN loaded (quality='high' enabled)")
except Exception as e:
    print(f"ℹ GFPGAN not available, quality='high' will skip restoration: {e}")


def _largest_face(faces):
    return max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))


def swap_faces(source_path, target_path, output_path, quality='medium', swap_type='face_swap'):
    """
    Replace the face in target_path with the face from source_path.
    quality:   'low' | 'medium' -> raw inswapper output
               'high'           -> inswapper + GFPGAN restoration (if installed)
    swap_type: 'face_swap' is the only type this model supports.
               Body swapping needs a different model and is not implemented.
    """
    try:
        if swap_type != 'face_swap':
            raise ValueError(f"swap_type '{swap_type}' not supported (inswapper only swaps faces)")

        source = cv2.imread(source_path)
        target = cv2.imread(target_path)
        if source is None:
            raise ValueError(f"Could not read source image: {source_path}")
        if target is None:
            raise ValueError(f"Could not read target image: {target_path}")

        src_faces = face_analyzer.get(source)
        tgt_faces = face_analyzer.get(target)
        if not src_faces:
            raise ValueError("No face detected in SOURCE image")
        if not tgt_faces:
            raise ValueError("No face detected in TARGET image")

        src_face = _largest_face(src_faces)
        tgt_face = _largest_face(tgt_faces)

        result = swapper.get(target, tgt_face, src_face, paste_back=True)

        if quality == 'high':
            if restorer is not None:
                _, _, result = restorer.enhance(result, has_aligned=False,
                                                only_center_face=False, paste_back=True)
            else:
                print("ℹ quality='high' requested but GFPGAN is not loaded; output not restored")

        cv2.imwrite(output_path, result)
        return True

    except Exception as e:
        print(f"ERROR in face swap: {e}")
        return False


print("✓ Face Swap Engine ready")

ONNX providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w60

In [ ]:
# SECTION 4: Main Processing Loop
import time
import uuid
from pathlib import Path

print("\n=== PROCESSING JOBS ===")
processed_count = 0

# Ensure required variables are defined as fallbacks if Section 2 wasn't run
if 'is_running' not in globals():
    is_running = False
    print("⚠ Warning: 'is_running' is not defined. Please run Section 2 first!")

if 'job_queue' not in globals():
    from collections import deque
    job_queue = deque()

while is_running:
    if job_queue:
        job = job_queue.popleft()
        job_id = job.get('job_id', str(uuid.uuid4()))

        try:
            # Send progress update
            progress_socket.send_json({
                'job_id': job_id,
                'progress': 10,
                'message': 'Downloading images from Drive...'
            })

            # Download images from Drive
            source_path = f"/tmp/{job_id}_source.jpg"
            target_path = f"/tmp/{job_id}_target.jpg"

            # Send progress: processing
            progress_socket.send_json({
                'job_id': job_id,
                'progress': 50,
                'message': 'Processing...'
            })

            # Swap faces
            output_path = f"/content/drive/MyDrive/ColabOutputs/{job_id}_result.jpg"
            success = swap_faces(
                source_path,
                target_path,
                output_path,
                quality=job.get('quality', 'medium'),
                swap_type=job.get('swap_type', 'face_swap')
            )

            if success:
                progress_socket.send_json({
                    'job_id': job_id,
                    'progress': 100,
                    'message': 'Complete!'
                })
                processed_count += 1
                print(f"✓ Job {job_id} completed")
            else:
                progress_socket.send_json({
                    'job_id': job_id,
                    'progress': 0,
                    'message': 'Error processing job'
                })
        except Exception as e:
            print(f"ERROR processing job {job_id}: {e}")
            progress_socket.send_json({
                'job_id': job_id,
                'progress': 0,
                'message': f'Error: {str(e)}'
            })

    time.sleep(1)

print(f"\nProcessed {processed_count} jobs")


=== PROCESSING JOBS ===
